# 第6章　久期与凸性

[![在 Colab 打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/albertandking/fixed-income/blob/main/notebooks/ch06_duration_convexity.ipynb) [![在 Binder 打开](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/albertandking/fixed-income/main?labpath=notebooks/ch06_duration_convexity.ipynb)

本 notebook 复现第6章正文中的全部数字：例6.1 / 例6.4 的久期、凸性、二阶近似，以及 6.8 的中国国债组合损益归因。


In [ ]:
# 自举单元：在 Colab/Binder 上自动安装本书复用包 fi；本地运行时自动跳过。
import importlib.util, sys, subprocess
if importlib.util.find_spec('fi') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/albertandking/fixed-income.git', '/content/fi-book'], check=False)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '/content/fi-book'], check=False)
    else:
        print('提示：请在仓库根目录执行 `uv sync` 后再运行本 notebook。')


In [ ]:
import numpy as np
from fi.cashflow import make_cashflows
from fi.pricing import price_bond
from fi import risk, plotting
plotting.use_chinese_style()


## 例6.1　3 年期附息国债的久期与凸性

面值 100、年付息（freq=1）、票息 3%、收益率 3%、剩余 3 年。


In [ ]:
cfs, ts = make_cashflows(coupon_rate=0.03, maturity=3, freq=1, face=100)
y = 0.03
P     = price_bond(cfs, ts, y, freq=1)
d_mac = risk.macaulay_duration(cfs, ts, y, freq=1)
d_mod = risk.modified_duration(cfs, ts, y, freq=1)
conv  = risk.convexity(cfs, ts, y, freq=1)
bpv   = risk.dv01(cfs, ts, y, freq=1)
print(f'价格 P            = {P:.4f}')
print(f'麦考利久期 D_mac  = {d_mac:.4f} 年')
print(f'修正久期   D_mod  = {d_mod:.4f}')
print(f'凸性       C      = {conv:.4f}')
print(f'DV01              = {bpv:.5f} 元/百元面值')


### 一阶 vs 二阶近似与真实重定价对照（例6.4）


In [ ]:
rows = []
for bp in (-100, -50, +50, +100):
    dy = bp / 10000
    true_pct = (price_bond(cfs, ts, y + dy, freq=1) / P - 1) * 100
    dur_pct  = (-d_mod * dy) * 100
    dc_pct   = risk.price_change(P, d_mod, conv, dy) / P * 100
    rows.append((bp, true_pct, dur_pct, dc_pct))

print(f"{'Δy(bp)':>7}{'真实%':>10}{'仅久期%':>10}{'久期+凸性%':>12}")
for bp, t, d, dc in rows:
    print(f'{bp:>7}{t:>10.3f}{d:>10.3f}{dc:>12.3f}')


### 价格—收益率曲线与久期切线（编程实验 6）

直观展示凸性带来的“涨多跌少”：真实曲线在切线之上。


In [ ]:
ys = np.linspace(0.00, 0.06, 121)
prices = [price_bond(cfs, ts, yi, freq=1) for yi in ys]
tangent = [P * (1 - d_mod * (yi - y)) for yi in ys]   # 仅久期的一阶近似

fig, ax = plotting.new_axes()
ax.plot(ys * 100, prices, label='真实价格 P(y)')
ax.plot(ys * 100, tangent, '--', label='久期切线（一阶近似）')
ax.scatter([y * 100], [P], color='k', zorder=5)
ax.set_xlabel('到期收益率 y (%)'); ax.set_ylabel('价格')
ax.set_title('图6-1　价格—收益率曲线与久期切线（凸性使真实价格高于切线）')
ax.legend()
fig.tight_layout()


## 6.8　中国国债组合损益归因

读取内置国债收益率曲线样本，构造 2Y/5Y/10Y 各 1000 万元市值的平价组合（票息=收益率，半年付息），
估算 +100bp 平行上行下的损益。


In [ ]:
from fi import data
curve = data.load_sample('cgb_yield_curve').set_index('tenor')['yield_pct'] / 100
curve[[2, 5, 10]]


In [ ]:
MV = 1000.0          # 每只市值（万元）
DY = 0.01            # +100bp
book = []
for tenor in (2, 5, 10):
    yi = float(curve[tenor])
    cf, t = make_cashflows(coupon_rate=yi, maturity=tenor, freq=2, face=100)
    dmod = risk.modified_duration(cf, t, yi, freq=2)
    cx   = risk.convexity(cf, t, yi, freq=2)
    dv   = dmod * MV * 1e-4                       # 万元/bp
    first = -dmod * MV * DY                       # 一阶损益（万元）
    cxadj = 0.5 * cx * MV * DY ** 2               # 凸性修正（万元）
    book.append(dict(tenor=tenor, ytm=yi, d_mod=dmod, conv=cx, mv=MV,
                     dv01=dv, first=first, cx_adj=cxadj, est=first + cxadj))

import pandas as pd
df = pd.DataFrame(book).set_index('tenor')
tot = df[['mv', 'dv01', 'first', 'cx_adj', 'est']].sum()
port_dmod = (df['d_mod'] * df['mv']).sum() / df['mv'].sum()
print(df.round(3))
print('\n组合修正久期 =', round(port_dmod, 3))
print('组合 DV01    =', round(tot['dv01'], 3), '万元/bp')
print('一阶损益     =', round(tot['first'], 1), '万元')
print('估计损益     =', round(tot['est'], 1), '万元')


### 10Y 国债的关键利率久期（编程实验 7）

在 2Y/5Y/10Y 三个节点单独平移零息曲线，验证 KRD 之和 ≈ 有效久期。


In [ ]:
key_tenors = np.array([2.0, 5.0, 10.0])
zeros = curve[[2, 5, 10]].to_numpy()                 # 用样本曲线作为零息曲线近似
cf10, t10 = make_cashflows(coupon_rate=float(curve[10]), maturity=10, freq=2, face=100)
krd = risk.key_rate_durations(cf10, t10, key_tenors, zeros)
for k, v in zip(key_tenors, krd):
    print(f'KRD({k:>4.0f}Y) = {v:.4f}')
print('KRD 之和   =', round(krd.sum(), 4))

fig, ax = plotting.new_axes(figsize=(7, 4))
ax.bar([f'{int(k)}Y' for k in key_tenors], krd)
ax.set_ylabel('关键利率久期'); ax.set_title('图6-2　10Y 国债的关键利率久期分布')
fig.tight_layout()


---

> 小结：`fi.risk` 的解析指标（例6.1）、二阶近似（例6.4）与组合归因（6.8）均可一键复现。
> 把普通债换成含权债时，改用 `effective_duration` / `effective_convexity` 即可，见第10章。
